# Deploy a saved Q-policy with logistic regression

Same **Adult Income** split and budgets as training, but swap the downstream
classifier to **logistic regression** while reusing the fitted Q-models from
`artifacts/fitted_q_policy.joblib`.

The Q-policy was trained against a different downstream model (typically
HistGradientBoosting). Here we treat it as a fixed acquisition policy and
measure how well it transfers when the classifier changes.

**Prerequisite:** train and save a policy first (`notebooks/adult_fitted_q_learning_py.py`
or the training cells in the main analysis notebook). Restart the kernel after
editing `adult_prototype_fitted_q_learning.py`.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import adult_prototype_fitted_q_learning

importlib.reload(adult_prototype_fitted_q_learning)

from adult_prototype_fitted_q_learning import (
    DEFAULT_Q_POLICY_ARTIFACT_PATH,
    baseline_policy_curves,
    build_experiment,
    downstream_model_settings,
    load_q_policy,
    metric_from_config,
    metric_label,
    plot_episode_step_diagnostics,
    run_final_episode,
    run_greedy_episode_with_budgets,
    score_on_test,
)

In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / DEFAULT_Q_POLICY_ARTIFACT_PATH

# Swap the downstream classifier used during rollout (Q-policy stays fixed).
DEPLOYMENT_DOWNSTREAM_MODEL = {
    "type": "logistic_regression",
    "max_iter": 100,
    "C": 5.0,
}
DEPLOYMENT_DOWNSTREAM_MODEL = {
    "type": "hist_gradient_boosting",
    "learning_rate": 0.01,
    "max_iter": 500,
    "max_leaf_nodes": 15,
    "min_samples_leaf": 5,
    "max_depth": 8,
    # optional: l2_regularization, max_bins, early_stopping,
    #           validation_fraction, n_iter_no_change
}

bundle = load_q_policy(ARTIFACT_PATH)
q_models = bundle["q_models"]
q_model_is_fitted = bundle["q_model_is_fitted"]
training_config = bundle["config"]
lagrangian_lambdas_by_budget = bundle.get("lagrangian_lambdas_by_budget")

rollout_config = {
    **training_config,
    "downstream_model": {
        **DEPLOYMENT_DOWNSTREAM_MODEL,
        "random_state": training_config["random_seed"],
    },
}

experiment = build_experiment(rollout_config)
config = experiment["config"]

fitted_actions = [action for action, fitted in q_model_is_fitted.items() if fitted]
print(f"Loaded artifact: {ARTIFACT_PATH}")
print(
    "Rollout budgets: "
    f"acquire={config['acquisition_budget']}, "
    f"retrain={config['retrain_budget']}, "
    f"evaluate={config['evaluation_budget']}"
)
print(f"Downstream model: {downstream_model_settings(config)[0]}")
print(f"Fitted Q-models: {fitted_actions}")

## Greedy rollout (held-out budgets)

Runs the saved Q-policy with epsilon = 0. State normalization still uses
`training_budget_maxima` from the saved config.

In [ ]:
final_result = run_final_episode(
    q_models,
    q_model_is_fitted,
    experiment,
    lagrangian_lambdas_by_budget=lagrangian_lambdas_by_budget,
)
test_scores = score_on_test(final_result["model"], experiment, config)

introspection_metric = metric_from_config(config, "introspection")
metric_name = metric_label(introspection_metric)

baseline_curves = baseline_policy_curves(experiment, config)
comparison_rows = [
    {
        "policy": "fitted Q + logistic regression",
        f"hidden_val_{introspection_metric}": final_result["terminal_score"],
        f"test_{introspection_metric}": test_scores.get(
            f"test_{introspection_metric}"
        ),
    },
    {
        "policy": "acquire-only baseline",
        f"hidden_val_{introspection_metric}": baseline_curves["acquire_only"][
            "introspection_score"
        ].iloc[-1],
        f"test_{introspection_metric}": None,
    },
    {
        "policy": "acquire-retrain baseline",
        f"hidden_val_{introspection_metric}": baseline_curves["acquire_retrain"][
            "introspection_score"
        ].iloc[-1],
        f"test_{introspection_metric}": None,
    },
]
comparison_table = pd.DataFrame(comparison_rows)

print(
    "Action sequence: "
    + " → ".join(step["action"] for step in final_result["action_history"])
)
print(f"Terminal hidden-validation {metric_name}: {final_result['terminal_score']:.4f}")
display(comparison_table)

## Step diagnostics with baselines

Edit the **configuration** in the next cell (budgets, batch size, introspection
metric/set, baselines on/off), then run it to roll out the saved Q-policy and plot.

In [ ]:
# --- episode step diagnostics configuration ---
DIAGNOSTICS_ACQUISITION_BUDGET = 10
DIAGNOSTICS_RETRAIN_BUDGET = 2
DIAGNOSTICS_EVALUATION_BUDGET = 5

# None keeps the batch fraction from the saved training config.
DIAGNOSTICS_BATCH_FRACTION = 0.001

# Plot introspection curve on this hold-out split and metric.
DIAGNOSTICS_INTROSPECTION_EVAL_SET = "hidden_validation"  # or "evaluation"
DIAGNOSTICS_INTROSPECTION_METRIC = None  # None -> config["introspection_metric"]

# Flush pending acquired rows with one final Retrain at episode end.
DIAGNOSTICS_FINAL_RETRAIN_PENDING_ROWS = config.get(
    "final_retrain_pending_rows", True
)

# Gray/orange baseline curves on the top panel.
DIAGNOSTICS_SHOW_BASELINES = True

# Optional seed override; None uses random_seed + final_episode_seed_offset.
DIAGNOSTICS_EPISODE_SEED = None

diagnostics_config = {
    **config,
    "acquisition_budget": DIAGNOSTICS_ACQUISITION_BUDGET,
    "retrain_budget": DIAGNOSTICS_RETRAIN_BUDGET,
    "evaluation_budget": DIAGNOSTICS_EVALUATION_BUDGET,
    "introspection_eval_set": DIAGNOSTICS_INTROSPECTION_EVAL_SET,
    "final_retrain_pending_rows": DIAGNOSTICS_FINAL_RETRAIN_PENDING_ROWS,
}
if DIAGNOSTICS_INTROSPECTION_METRIC is not None:
    diagnostics_config["introspection_metric"] = DIAGNOSTICS_INTROSPECTION_METRIC
if DIAGNOSTICS_BATCH_FRACTION is not None:
    diagnostics_config["batch_fraction"] = DIAGNOSTICS_BATCH_FRACTION
    diagnostics_config["batch_size"] = max(
        1, int(round(DIAGNOSTICS_BATCH_FRACTION * diagnostics_config["total_rows"]))
    )

diagnostics_experiment = {**experiment, "config": diagnostics_config}

diagnostics_result = run_greedy_episode_with_budgets(
    q_models=q_models,
    q_model_is_fitted=q_model_is_fitted,
    experiment=diagnostics_experiment,
    acquisition_budget=diagnostics_config["acquisition_budget"],
    retrain_budget=diagnostics_config["retrain_budget"],
    evaluation_budget=diagnostics_config["evaluation_budget"],
    episode_seed=DIAGNOSTICS_EPISODE_SEED,
    lagrangian_lambdas_by_budget=lagrangian_lambdas_by_budget,
)

introspection_metric = metric_from_config(diagnostics_config, "introspection")
print(
    "Action sequence: "
    + " → ".join(step["action"] for step in diagnostics_result["action_history"])
)
print(
    f"Budgets: acquire={diagnostics_config['acquisition_budget']}, "
    f"retrain={diagnostics_config['retrain_budget']}, "
    f"evaluate={diagnostics_config['evaluation_budget']}"
)
if DIAGNOSTICS_BATCH_FRACTION is not None:
    print(
        f"Batch fraction={DIAGNOSTICS_BATCH_FRACTION:.4f}; "
        f"rows per Acquire={diagnostics_config['batch_size']}"
    )
print(
    f"Terminal {diagnostics_config['introspection_eval_set'].replace('_', ' ')} "
    f"{metric_label(introspection_metric)}: "
    f"{diagnostics_result['terminal_score']:.4f}"
)

plot_episode_step_diagnostics(
    diagnostics_result["step_diagnostics"],
    diagnostics_config,
    experiment=diagnostics_experiment if DIAGNOSTICS_SHOW_BASELINES else None,
)